# 05 — Cross-Generator Generalisation Evaluation

**Milestone 4** (July 6–20, 2026)

This notebook evaluates the CIFAKE-trained EfficientNet-B3 detector on images from four
progressively newer generator families:

| Family | Source | Type |
|--------|--------|------|
| StyleGAN | ForenSynths / CNNDetection | GAN (older) |
| SD3/Flux | GenImage++ | Modern diffusion |
| Midjourney v6 | CortexLM/midjourney-v6 | Proprietary diffusion |
| GPT-4o | Yejy53/GPT-ImgEval | Autoregressive hybrid |

**Research hypothesis:** The detector relies on Stable Diffusion v1.4-specific texture
artifacts and will show progressive degradation on newer generator families.

**Prerequisites:** Upload the CIFAKE zip file to `My Drive/ai-image-detection/` before running the setup cells below.

## 0. CIFAKE Setup

In [ ]:
# Cell 1 — Mount Google Drive & define paths
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
REPO_DIR = "/content/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone the repo (update URL to your own fork)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Extract CIFAKE from zip uploaded to Google Drive
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, "train")):
    print("CIFAKE already extracted — skipping.")
else:
    # Find the zip file in the Drive folder (handles any filename)
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            f"No zip file found in {DRIVE_ROOT}. "
            "Upload the CIFAKE zip downloaded from Kaggle to that folder."
        )

    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print("Extracting (this takes ~1-2 minutes)...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)

    print("Extraction complete.")

# Show what was extracted so we can verify the folder structure
extracted = os.listdir(DATA_DIR)
print(f"Contents of DATA_DIR: {extracted}")

# If zip extracted into a nested subfolder, adjust DATA_DIR automatically
if "train" not in extracted and len(extracted) == 1:
    DATA_DIR = os.path.join(DATA_DIR, extracted[0])
    print(f"Adjusted DATA_DIR to: {DATA_DIR}")

print(f"Final DATA_DIR contents: {os.listdir(DATA_DIR)}")

In [ ]:
# Cell 4 — Verify dataset splits
import os

splits = {}
for split in ["train", "test"]:
    for cls in ["REAL", "FAKE"]:
        path = os.path.join(DATA_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        splits[f"{split}/{cls}"] = count

print("CIFAKE Dataset Summary")
print("=" * 30)
for key, count in splits.items():
    print(f"  {key:15s} : {count:,}")
print(f"  {'Total':15s} : {sum(splits.values()):,}")

assert splits["train/REAL"] == 50_000 or splits["train/REAL"] == 30_000, (
    f"Unexpected train/REAL count: {splits['train/REAL']}"
)

## 1. Setup

In [ ]:
# Bridge: connect CIFAKE paths to the rest of this notebook
import sys

IN_COLAB = 'google.colab' in sys.modules
DRIVE_BASE = DRIVE_ROOT
PROJECT_ROOT = REPO_DIR
CIFAKE_DATA_DIR = DATA_DIR
REAL_REFERENCE_DIR = os.path.join(CIFAKE_DATA_DIR, "test", "REAL")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"REAL reference dir: {REAL_REFERENCE_DIR}")
print(f"Exists: {os.path.exists(REAL_REFERENCE_DIR)}")

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.model.detector import build_detector, load_checkpoint
from src.utils.data_loader import (
    get_generalisation_loader,
    discover_generator_families,
)
from src.evaluation.generalisation import (
    evaluate_generator,
    compute_degradation,
    save_generalisation_results,
    plot_cross_generator_accuracy,
    plot_cross_generator_auc,
    plot_degradation_waterfall,
    plot_confidence_distributions,
    plot_ece_comparison,
)
from src.explainability.gradcam import (
    batch_gradcam_failures,
    comparative_grid,
    write_attention_notes,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# --- Load trained model ---
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_detector.pth')
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = os.path.join(REPO_DIR, 'outputs/checkpoints/best_detector.pth')

model = build_detector(num_classes=2)
model = load_checkpoint(model, CHECKPOINT_PATH)
model = model.to(DEVICE)
model.eval()
print(f"Model loaded from: {CHECKPOINT_PATH}")

# Temperature from baseline calibration
TEMPERATURE = 1.2189

# Load baseline results
BASELINE_PATH = os.path.join(DRIVE_BASE, 'outputs/results/baseline_results.json')
if not os.path.exists(BASELINE_PATH):
    BASELINE_PATH = os.path.join(REPO_DIR, 'outputs/results/baseline_results.json')

with open(BASELINE_PATH) as f:
    baseline_results = json.load(f)

print(f"Baseline accuracy: {baseline_results['test_accuracy']:.4f}")
print(f"Baseline AUC: {baseline_results['test_auc']:.4f}")

## 2. Data Acquisition

Download images from each generator family using streaming mode.
This is idempotent — re-running skips families that already have enough images.

In [ ]:
from scripts.download_generalisation_data import download_all_families

# Configuration
PER_GENERATOR = 300
GEN_DATA_DIR = os.path.join(DRIVE_BASE, 'data/generalisation')

download_results = download_all_families(
    output_dir=GEN_DATA_DIR,
    per_generator=PER_GENERATOR,
    seed=SEED,
)

In [ ]:
# Verify what we have
families = discover_generator_families(GEN_DATA_DIR)
print(f"\nAvailable generator families ({len(families)}):")
for fdir in families:
    fake_dir = Path(fdir) / 'FAKE'
    real_dir = Path(fdir) / 'REAL'
    n_fake = len(list(fake_dir.glob('*'))) if fake_dir.exists() else 0
    n_real = len(list(real_dir.glob('*'))) if real_dir.exists() else 0
    print(f"  {Path(fdir).name:20s} | FAKE: {n_fake:4d} | REAL: {n_real:4d}")

## 3. Per-Generator Evaluation

In [ ]:
# Evaluate each family (REAL_REFERENCE_DIR set in bridge cell above)
all_results = {}
all_degradation = {}

for fdir in families:
    family_name = Path(fdir).name
    print(f"\n{'='*60}")
    print(f"Evaluating: {family_name}")
    print(f"{'='*60}")
    
    # Check if family has matched real images
    has_matched_real = (Path(fdir) / 'REAL').exists()
    
    loader = get_generalisation_loader(
        family_dir=fdir,
        real_reference_dir=REAL_REFERENCE_DIR,
        batch_size=32,
        matched_real=has_matched_real,
    )
    
    results = evaluate_generator(
        model=model,
        loader=loader,
        device=DEVICE,
        temperature=TEMPERATURE,
        family_name=family_name,
    )
    all_results[family_name] = results
    
    deg = compute_degradation(baseline_results, results)
    all_degradation[family_name] = deg
    
    print(f"  Accuracy:            {results['accuracy']:.4f}")
    print(f"  Fake detection rate: {results['fake_detection_rate']:.4f}")
    print(f"  AUC:                 {results['auc']}")
    print(f"  F1 (FAKE):           {results['f1_fake']:.4f}")
    print(f"  ECE (pre-T):         {results['ece_pre_calibration']:.4f}")
    print(f"  ECE (post-T):        {results['ece_post_calibration']:.4f}")
    print(f"  Drop from baseline:  {deg['accuracy_drop_absolute']:.4f} "
          f"({deg['accuracy_drop_relative']:.1%} relative)")

## 4. Degradation Analysis

In [ ]:
# Summary table
summary_data = []
for fam, res in all_results.items():
    deg = all_degradation[fam]
    summary_data.append({
        'Generator': fam,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Fake Det. Rate': f"{res['fake_detection_rate']:.4f}" if res['fake_detection_rate'] else 'N/A',
        'AUC': f"{res['auc']:.4f}" if res['auc'] else 'N/A',
        'F1 (FAKE)': f"{res['f1_fake']:.4f}",
        'ECE (pre-T)': f"{res['ece_pre_calibration']:.4f}",
        'ECE (post-T)': f"{res['ece_post_calibration']:.4f}",
        'Acc. Drop (abs)': f"{deg['accuracy_drop_absolute']:.4f}",
        'Acc. Drop (rel)': f"{deg['accuracy_drop_relative']:.1%}",
    })

# Add baseline row
summary_data.insert(0, {
    'Generator': 'CIFAKE (baseline)',
    'Accuracy': f"{baseline_results['test_accuracy']:.4f}",
    'Fake Det. Rate': '—',
    'AUC': f"{baseline_results['test_auc']:.4f}",
    'F1 (FAKE)': '—',
    'ECE (pre-T)': f"{baseline_results['ece_before_calibration']:.4f}",
    'ECE (post-T)': f"{baseline_results['ece_after_calibration']:.4f}",
    'Acc. Drop (abs)': '0.0000',
    'Acc. Drop (rel)': '0.0%',
})

df_summary = pd.DataFrame(summary_data)
df_summary.style.set_caption('Cross-Generator Generalisation Results')

In [ ]:
# Save results
RESULTS_DIR = 'outputs/results'
if IN_COLAB:
    RESULTS_DIR = os.path.join(DRIVE_BASE, 'outputs/results')

save_generalisation_results(all_results, all_degradation, RESULTS_DIR)
print("Results saved.")

## 5. Grad-CAM Failure Analysis

In [ ]:
# Target layer for Grad-CAM: last conv block of EfficientNet-B3
target_layer = model.conv_head

HEATMAP_DIR = 'outputs/heatmaps'
PLOTS_DIR = 'outputs/plots'
if IN_COLAB:
    HEATMAP_DIR = os.path.join(DRIVE_BASE, 'outputs/heatmaps')
    PLOTS_DIR = os.path.join(DRIVE_BASE, 'outputs/plots')

family_names = []

for fdir in families:
    family_name = Path(fdir).name
    family_names.append(family_name)
    print(f"\nGenerating Grad-CAM for: {family_name}")
    
    # Use fake-only loader for Grad-CAM (we want to analyze failures on fakes)
    loader = get_generalisation_loader(
        family_dir=fdir,
        real_reference_dir=None,  # fake-only
        batch_size=16,
    )
    
    batch_gradcam_failures(
        model=model,
        loader=loader,
        target_layer=target_layer,
        family=family_name,
        device=DEVICE,
        k=16,
        output_dir=HEATMAP_DIR,
    )

In [ ]:
# Comparative grid
comparative_grid(
    families=family_names,
    heatmap_dir=HEATMAP_DIR,
    n_per_family=4,
    save_path=os.path.join(PLOTS_DIR, 'gradcam_comparison_grid.png'),
)

# Attention notes template
write_attention_notes(family_names, output_dir=HEATMAP_DIR)

In [ ]:
# Display the comparison grid
grid_path = os.path.join(PLOTS_DIR, 'gradcam_comparison_grid.png')
if os.path.exists(grid_path):
    from IPython.display import Image, display
    display(Image(filename=grid_path))

## 6. Visualisations

In [ ]:
# Cross-generator accuracy bar chart
plot_cross_generator_accuracy(
    all_results,
    baseline_results['test_accuracy'],
    os.path.join(PLOTS_DIR, 'cross_generator_accuracy.png'),
)

# Cross-generator AUC bar chart
plot_cross_generator_auc(
    all_results,
    baseline_results['test_auc'],
    os.path.join(PLOTS_DIR, 'cross_generator_auc.png'),
)

# Degradation waterfall
plot_degradation_waterfall(
    all_degradation,
    baseline_results['test_accuracy'],
    os.path.join(PLOTS_DIR, 'degradation_waterfall.png'),
)

# Confidence distributions
plot_confidence_distributions(
    all_results,
    os.path.join(PLOTS_DIR, 'confidence_distributions_by_generator.png'),
)

# ECE comparison
plot_ece_comparison(
    all_results,
    os.path.join(PLOTS_DIR, 'ece_comparison_by_generator.png'),
)

print("All plots saved.")

In [ ]:
# Display key plots inline
from IPython.display import Image, display

for plot_name in [
    'cross_generator_accuracy.png',
    'degradation_waterfall.png',
    'confidence_distributions_by_generator.png',
    'ece_comparison_by_generator.png',
]:
    plot_path = os.path.join(PLOTS_DIR, plot_name)
    if os.path.exists(plot_path):
        print(f"\n--- {plot_name} ---")
        display(Image(filename=plot_path))

## 7. Summary Table

Final formatted table for the progress report.

In [ ]:
# Final styled summary table
display(df_summary.style
    .set_caption('Cross-Generator Generalisation Evaluation Results')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '14px'), ('font-weight', 'bold')]
    }])
)

In [ ]:
# Export summary as LaTeX for the report
print("\n--- LaTeX table for progress report ---\n")
print(df_summary.to_latex(index=False, escape=True))

## 8. Discussion

<!-- TODO: Fill in after running the evaluation -->

### Key Findings

TODO: Summarise the main results:
- Which generator family showed the least/most degradation?
- Does the degradation order match the hypothesis?
- How does calibration (ECE) behave under distribution shift?

### Grad-CAM Observations

TODO: Describe the qualitative patterns observed:
- What does the model attend to on false negatives from each family?
- How do attention patterns differ between generators?
- What artifacts are present/absent across families?

### Implications

TODO: Connect findings to the research question:
- What does this tell us about the generalisation gap?
- Which features transfer and which are generator-specific?
- Recommendations for more robust detection

In [ ]:
# Final save: ensure all JSON results are persisted
print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"\nResults: {RESULTS_DIR}/")
print(f"Plots:   {PLOTS_DIR}/")
print(f"Heatmaps: {HEATMAP_DIR}/")
print(f"\nFiles produced:")
for f in ['generalisation_results.json', 'degradation_summary.json']:
    p = os.path.join(RESULTS_DIR, f)
    print(f"  {'✓' if os.path.exists(p) else '✗'} {f}")
for f in ['cross_generator_accuracy.png', 'cross_generator_auc.png',
          'degradation_waterfall.png', 'confidence_distributions_by_generator.png',
          'ece_comparison_by_generator.png', 'gradcam_comparison_grid.png']:
    p = os.path.join(PLOTS_DIR, f)
    print(f"  {'✓' if os.path.exists(p) else '✗'} {f}")